# 02 - Routing: chèn SWAP, và heuristic mà RL phải đánh bại

Notebook `01` cho thấy layout tốt có thể làm routing biến mất. Nhưng với mạch thật thì interaction
graph luôn dày hơn coupling map, nên **không có** layout nào đủ tốt - compiler buộc phải chèn SWAP.

Đây là notebook trọng tâm của folder, vì **qubit routing là bài toán biên dịch được nghiên cứu nhiều
nhất bằng reinforcement learning**. Lý do: nó là một bài toán quyết định tuần tự rời rạc, có baseline
mạnh và dễ đo (`SabreSwap`), và cải thiện được thì mọi mạch chạy trên thiết bị đều hưởng lợi.

Mục tiêu của notebook:

1. Hiểu vòng lặp routing, và **tự cài đặt** một router từ đầu (`mini_router`) - vì không viết được
   router thì không viết được environment.
2. Đọc **công thức heuristic của SABRE** - đó là đối thủ cụ thể.
3. Biết cách **kiểm chứng** output: hợp lệ (`CheckMap`) và đúng (hoán vị).
4. Biết routing nhạy với `seed` và topology đến mức nào, để không báo cáo số liệu sai.

In [ ]:
import numpy as np
import pandas as pd

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.random import random_circuit
from qiskit.converters import circuit_to_dag
from qiskit.quantum_info import Operator
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.passes import (
    ApplyLayout,
    BasicSwap,
    CheckMap,
    ElidePermutations,
    EnlargeWithAncilla,
    FullAncillaAllocation,
    LookaheadSwap,
    OptimizeSwapBeforeMeasure,
    SabreLayout,
    SabreSwap,
    TrivialLayout,
)

BASIS = ["cx", "rz", "sx", "x"]


def mach_thu(n, do_sau, seed):
    """Mạch ngẫu nhiên đã dịch về basis gates, chưa route."""
    qc = random_circuit(n, do_sau, max_operands=2, seed=seed)
    return transpile(qc, basis_gates=BASIS, optimization_level=0)


def route(qc, cm, router=None, layout=None, seed=0):
    """Công thức chuẩn từ notebook 01: layout -> ancilla -> apply -> route -> kiểm tra."""
    router = router if router is not None else SabreSwap(cm, heuristic="decay", seed=seed)
    layout = layout if layout is not None else TrivialLayout(cm)
    pm = PassManager([layout, FullAncillaAllocation(cm), EnlargeWithAncilla(),
                      ApplyLayout(), router, CheckMap(cm)])
    out = pm.run(qc)
    return out, pm.property_set


CM5 = CouplingMap.from_line(5)
mach = mach_thu(5, 10, seed=11)
print("mạch thử: 5 qubit, cx =", mach.count_ops().get("cx"), ", depth =", mach.depth())
print("coupling map: line(5), distance(0,4) =", int(CM5.distance_matrix[0][4]))

## 1. Vòng lặp routing

Mọi router - từ `BasicSwap` tới SABRE tới một RL policy - đều là cùng một vòng lặp, chỉ khác nhau ở
**một dòng**:

```
pi = layout ban đầu                      # pi[qubit logic] = qubit vật lý
while DAG còn cổng:
    thực thi mọi cổng ở front layer mà pi làm cho hợp lệ
    if không thực thi được cổng nào:
        chọn một edge (a, b) của coupling map       <--- DÒNG DUY NHẤT CẦN QUYẾT ĐỊNH
        chèn SWAP(a, b), cập nhật pi
```

Vòng lặp này luôn kết thúc (vì luôn có SWAP đưa hai qubit lại gần nhau hơn), nhưng **số SWAP phụ
thuộc hoàn toàn vào dòng được tô đậm**. Tối thiểu hoá nó là NP-hard, nên mọi thứ dùng trong thực tế
đều là heuristic.

Ta tự cài vòng lặp này. Quyết định chọn SWAP dùng heuristic đơn giản nhất của SABRE, gọi là
**basic**: chọn SWAP làm giảm nhiều nhất tổng khoảng cách của các cổng đang nằm ở front layer.

$$H_{\text{basic}}(\pi) = \sum_{(q_a, q_b) \,\in\, F} D\big[\pi(q_a)\big]\big[\pi(q_b)\big]$$

với $F$ là front layer và $D$ là distance matrix. Cổng hợp lệ có $D = 1$, nên $H$ nhỏ nghĩa là "các
cổng cần làm ngay đang ở gần nhau".

In [ ]:
def mini_router(qc, cm):
    """Router tự viết: vòng lặp front layer + heuristic basic của SABRE.

    Trả về (mạch đã route, pi cuối cùng, số SWAP).
    pi[qubit logic] = qubit vật lý mà nó đang nằm.
    """
    D = cm.distance_matrix
    dag = circuit_to_dag(qc)
    n = qc.num_qubits
    pi = list(range(n))                      # layout ban đầu: tầm thường
    out = QuantumCircuit(n)
    out.global_phase = qc.global_phase
    edges = sorted(set(tuple(sorted(e)) for e in cm.get_edges()))
    so_swap = 0

    while dag.op_nodes():
        # (1) thực thi mọi cổng ở front layer mà pi làm cho hợp lệ
        tien_do = False
        for node in list(dag.front_layer()):
            qs = [dag.find_bit(q).index for q in node.qargs]
            if len(qs) == 1 or D[pi[qs[0]]][pi[qs[1]]] == 1:
                out.append(node.op, [pi[q] for q in qs])
                dag.remove_op_node(node)
                tien_do = True
        if tien_do:
            continue

        # (2) bế tắc -> phải chèn SWAP. Đây là chỗ cần quyết định.
        cap_can = [[dag.find_bit(q).index for q in nd.qargs]
                   for nd in dag.front_layer() if len(nd.qargs) == 2]

        def H(p):
            return sum(D[p[a]][p[b]] for a, b in cap_can)

        # chỉ xét SWAP có liên quan tới qubit đang bị kẹt
        dang_ket = {pi[q] for cap in cap_can for q in cap}
        ung_vien = [e for e in edges if e[0] in dang_ket or e[1] in dang_ket]

        tot_nhat, H_tot = None, None
        for (a, b) in ung_vien:
            p2 = list(pi)
            la, lb = p2.index(a), p2.index(b)
            p2[la], p2[lb] = p2[lb], p2[la]
            h = H(p2)
            if H_tot is None or h < H_tot:
                tot_nhat, H_tot = (a, b), h

        la, lb = pi.index(tot_nhat[0]), pi.index(tot_nhat[1])
        pi[la], pi[lb] = pi[lb], pi[la]
        out.swap(*tot_nhat)
        so_swap += 1

    return out, pi, so_swap


tu_viet, pi_cuoi, n_swap = mini_router(mach, CM5)
print(f"mini_router: {n_swap} swap, depth {tu_viet.depth()}, pi cuối = {pi_cuoi}")

sabre_basic, ps = route(mach, CM5, router=SabreSwap(CM5, heuristic="basic", seed=0))
print(f"SabreSwap basic: {sabre_basic.count_ops().get('swap', 0)} swap, depth {sabre_basic.depth()}")
print(f"\n-> Cài đúng công thức rồi vẫn thua Qiskit. Vì SabreSwap còn làm thêm nhiều thứ:")
print("   lọc ứng viên kỹ hơn, chạy mạch cả hai chiều để chọn layout, và dùng heuristic tốt hơn.")

Kết quả này quan trọng về mặt phương pháp: **cài đúng công thức trong paper vẫn chưa đủ để bằng
implementation thật**. Nếu bạn so RL agent với `mini_router` thì agent sẽ "thắng" dễ dàng và con số
đó vô nghĩa. Baseline phải là `SabreSwap` của Qiskit, chạy nhiều seed.

## 2. Kiểm chứng output: hai câu hỏi khác nhau

Một router có thể sai theo hai cách độc lập:

| Câu hỏi | Công cụ | Nếu bỏ qua |
|---|---|---|
| Mạch có **hợp lệ** với coupling map? | `CheckMap` → `property_set["is_swap_mapped"]` | agent "thắng" bằng cách phớt lờ cổng khó |
| Mạch có **đúng** (cùng unitary)? | so sánh có bù hoán vị | agent "thắng" bằng cách làm sai |

Câu thứ hai có bẫy. Mạch route ra chứa SWAP, nên cuối mạch qubit logic $i$ không còn nằm ở qubit vật
lý $i$ nữa. Có hai cách bù lại, và **chiều của hoán vị rất dễ nhầm**:

- **Cách chắc chắn:** chèn thêm SWAP vào cuối mạch để đưa mọi qubit về chỗ cũ, rồi so sánh trực tiếp.
- **Cách gọn:** `Operator(ket_qua).equiv(Operator(goc).apply_permutation(inv))`, với
  `inv[vật lý] = logic`, tức là **nghịch đảo** của `pi`. Dùng chính `pi` sẽ cho `False` ngay khi
  hoán vị không phải một phép đổi chỗ đơn lẻ.

In [ ]:
def kiem_tra_dung(goc, ket_qua, pi):
    """Hai cách bù hoán vị, cả hai phải cho cùng kết quả."""
    n = goc.num_qubits
    U_goc = Operator(goc)

    # cách 1: chèn SWAP bù vào cuối
    bu = ket_qua.copy()
    p = list(pi)
    for logic in range(n):
        if p[logic] != logic:
            khac = p.index(logic)
            bu.swap(p[logic], logic)
            p[logic], p[khac] = p[khac], p[logic]
    cach1 = Operator(bu).equiv(U_goc)

    # cách 2: apply_permutation với NGHỊCH ĐẢO của pi
    inv = [pi.index(i) for i in range(n)]
    cach2 = Operator(ket_qua).equiv(U_goc.apply_permutation(inv))

    # cách SAI thường gặp: dùng chính pi, hoặc không bù gì cả
    sai1 = Operator(ket_qua).equiv(U_goc.apply_permutation(list(pi)))
    sai2 = Operator(ket_qua).equiv(U_goc)
    return cach1, cach2, sai1, sai2


pm = PassManager([CheckMap(CM5)])
pm.run(tu_viet)
print("hợp lệ (CheckMap):", pm.property_set["is_swap_mapped"])

c1, c2, s1, s2 = kiem_tra_dung(mach, tu_viet, pi_cuoi)
print(f"\npi cuối = {pi_cuoi}  (không phải một phép đổi chỗ đơn lẻ)")
print(f"  đúng - chèn SWAP bù vào cuối      : {c1}")
print(f"  đúng - apply_permutation(nghịch đảo): {c2}")
print(f"  SAI  - apply_permutation(pi)        : {s1}")
print(f"  SAI  - không bù hoán vị             : {s2}")

## 3. Ba heuristic của SABRE

`SabreSwap` nhận tham số `heuristic` với ba giá trị, là ba hàm cost khác nhau. Gọi $F$ là front layer
và $E$ là tập cổng "sắp tới" (extended set - vài lớp kế tiếp trong DAG):

**`basic`** - chỉ nhìn front layer:

$$H = \frac{1}{\lvert F \rvert}\sum_{g \in F} D\big[\pi(g_1)\big]\big[\pi(g_2)\big]$$

**`lookahead`** - cộng thêm phần nhìn trước, có trọng số nhỏ hơn:

$$H = \frac{1}{\lvert F \rvert}\sum_{g \in F} D[\cdot] \;+\; W \cdot \frac{1}{\lvert E \rvert}\sum_{g \in E} D[\cdot]$$

**`decay`** - thêm hệ số phạt cho qubit vừa bị SWAP, để tránh dồn hết SWAP vào một qubit và làm
depth tăng vọt:

$$H_{\text{decay}} = \max\big(\text{decay}[\pi(a)], \text{decay}[\pi(b)]\big) \cdot H_{\text{lookahead}}$$

`decay` là mặc định. Đây chính là **hàm mục tiêu mà một RL policy cần vượt** - và lưu ý nó đã khá
tinh vi: nó cân bằng giữa số SWAP và depth, chứ không chỉ đếm SWAP.

In [ ]:
print(f"{'router':22s} {'swap':>5s} {'depth':>6s} {'hợp lệ':>7s}")
ket_qua = {}
for ten, r in [
    ("BasicSwap", BasicSwap(CM5)),
    ("LookaheadSwap", LookaheadSwap(CM5)),
    ("SabreSwap basic", SabreSwap(CM5, heuristic="basic", seed=0)),
    ("SabreSwap lookahead", SabreSwap(CM5, heuristic="lookahead", seed=0)),
    ("SabreSwap decay", SabreSwap(CM5, heuristic="decay", seed=0)),
]:
    out, ps = route(mach, CM5, router=r)
    sw = out.count_ops().get("swap", 0)
    ket_qua[ten] = sw
    print(f"  {ten:20s} {sw:5d} {out.depth():6d} {str(ps['is_swap_mapped']):>7s}")
print(f"  {'mini_router (tự viết)':20s} {n_swap:5d} {tu_viet.depth():6d}    True")

print("\n-> BasicSwap chỉ đưa qubit lại gần nhau theo đường ngắn nhất, không nhìn cổng tiếp theo:")
print(f"   tốn {ket_qua['BasicSwap']} swap, gấp {ket_qua['BasicSwap'] / ket_qua['SabreSwap decay']:.1f} lần decay.")
print("-> lookahead và decay bằng nhau ở mạch nhỏ này; khác biệt lộ ra ở mạch lớn hơn.")

## 4. Độ nhạy theo seed - lý do không bao giờ báo cáo một seed

SABRE phá thế hoà (nhiều SWAP cùng cost) bằng số ngẫu nhiên, nên kết quả phụ thuộc `seed`. Trên mạch
nhỏ thì thường không thấy, nhưng mạch lớn thì khoảng dao động là vài phần trăm - đủ để một "cải
thiện 3%" trở thành nhiễu thuần.

Đây là lỗi phương pháp phổ biến nhất trong các báo cáo RL-for-routing: so agent (chạy nhiều lần, lấy
tốt nhất) với baseline chạy **một** seed.

In [ ]:
CM9 = CouplingMap.from_line(9)
mach9 = mach_thu(9, 24, seed=3)
print("mạch 9 qubit: cx =", mach9.count_ops().get("cx"), "\n")

sw_swap = [route(mach9, CM9, router=SabreSwap(CM9, heuristic="decay", seed=s))[0]
           .count_ops().get("swap", 0) for s in range(15)]
print("SabreSwap (layout tầm thường), seed 0..14:")
print(" ", sw_swap)
print(f"  min={min(sw_swap)} max={max(sw_swap)} mean={np.mean(sw_swap):.1f} std={np.std(sw_swap):.2f}"
      f"  -> dao động {(max(sw_swap) - min(sw_swap)) / np.mean(sw_swap) * 100:.0f}% chỉ do seed")

sw_layout = []
for s in range(15):
    pm = PassManager([SabreLayout(CM9, seed=s), CheckMap(CM9)])
    o = pm.run(mach9)
    assert pm.property_set["is_swap_mapped"]
    sw_layout.append(o.count_ops().get("swap", 0))
print("\nSabreLayout (tự chọn layout + tự route), seed 0..14:")
print(" ", sw_layout)
print(f"  min={min(sw_layout)} max={max(sw_layout)} mean={np.mean(sw_layout):.1f} std={np.std(sw_layout):.2f}")

print(f"\n-> Chọn layout tốt giảm {(1 - np.mean(sw_layout) / np.mean(sw_swap)) * 100:.0f}% số SWAP")
print("   so với route trên layout tầm thường. Layout và routing KHÔNG tách rời được.")
print("-> Muốn so sánh công bằng: nhiều seed, báo cáo mean ± std, và nói rõ layout nào.")

Hai con số trên đáng ghi nhớ khi thiết kế thí nghiệm RL:

- Dao động do seed khoảng 7% → một agent muốn chứng minh hơn thật phải hơn **nhiều hơn** 7%, hoặc
  phải so trên đủ nhiều seed để std của trung bình nhỏ lại.
- SabreLayout (làm layout và routing **cùng lúc**) hơn hẳn routing trên layout tầm thường. Nên nếu
  agent của bạn chỉ làm routing với layout cố định, hãy so với `SabreSwap` ở **cùng** layout đó -
  so với `SabreLayout` là so sai bài toán.

## 5. Topology quyết định độ khó

Số edge của coupling map ảnh hưởng tới số SWAP mạnh hơn mọi lựa chọn heuristic nào.

In [ ]:
maps = {
    "line(9)": CouplingMap.from_line(9),
    "ring(9)": CouplingMap.from_ring(9),
    "grid(3,3)": CouplingMap.from_grid(3, 3),
    "full(9)": CouplingMap.from_full(9),
}

rows = []
for s in range(5):
    qc = mach_thu(9, 20, seed=s)
    for ten, cm in maps.items():
        out, ps = route(qc, cm, seed=s)
        assert ps["is_swap_mapped"]
        n_edge = len(set(tuple(sorted(e)) for e in cm.get_edges()))
        rows.append({"topology": ten, "edge": n_edge, "bậc": round(2 * n_edge / cm.size(), 1),
                     "swap": out.count_ops().get("swap", 0), "depth": out.depth()})

bang = pd.DataFrame(rows).groupby(["topology", "edge", "bậc"], sort=False)[["swap", "depth"]].mean()
print(bang.round(1).to_string())
print("\n-> line -> ring chỉ thêm MỘT edge mà số SWAP giảm 44%.")
print("-> full(9) không cần SWAP nào: routing chỉ tồn tại vì phần cứng thiếu kết nối.")
print("-> grid ít SWAP hơn ring nhưng depth lại cao hơn: hai thước đo KHÔNG cùng chiều.")

Dòng cuối quan trọng cho việc thiết kế reward: `grid(3,3)` ít SWAP hơn `ring(9)` nhưng depth cao hơn.
Nếu reward chỉ là "âm số SWAP" thì agent sẽ tối ưu một thứ mà đôi khi làm mạch chạy **lâu hơn**, tức
là mất coherence nhiều hơn. Notebook `05` quay lại chuyện này.

## 6. Vài pass phụ trợ đáng biết

| Pass | Việc |
|---|---|
| `ElidePermutations` | xoá SWAP **tường minh** trong mạch gốc bằng cách đổi nhãn qubit (miễn phí) |
| `OptimizeSwapBeforeMeasure` | SWAP ngay trước `measure` là vô nghĩa - đổi nhãn clbit là xong |
| `StarPreRouting` | xử lý riêng mạch có interaction graph hình sao |
| `Commuting2qGateRouter` | khi các cổng 2-qubit **giao hoán** với nhau (QAOA), thứ tự tự do hơn nhiều |

Hai pass đầu minh hoạ một ý quan trọng: **không phải SWAP nào cũng tốn tiền**. SWAP ở cuối mạch,
hoặc SWAP viết tường minh trong mạch gốc, chỉ là đổi tên qubit.

In [ ]:
qe = QuantumCircuit(4)
qe.h(0)
qe.swap(0, 3)
qe.cx(0, 1)
qe.swap(1, 2)
print("ElidePermutations")
print("  trước:", dict(qe.count_ops()))
pm = PassManager([ElidePermutations()])
print("  sau  :", dict(pm.run(qe).count_ops()), "-> 2 SWAP biến thành 0, hoán vị ghi vào final_layout")

qm = QuantumCircuit(2, 2)
qm.h(0)
qm.cx(0, 1)
qm.swap(0, 1)
qm.measure([0, 1], [0, 1])
print("\nOptimizeSwapBeforeMeasure")
print("  trước:", dict(qm.count_ops()))
print("  sau  :", dict(PassManager([OptimizeSwapBeforeMeasure()]).run(qm).count_ops()),
      "-> SWAP trước measure là miễn phí")

print("\n-> Khi tính reward cho RL, nếu đếm cả SWAP miễn phí thì agent bị phạt oan.")

## Tóm tắt

| Nội dung | Chi tiết |
|---|---|
| vòng lặp routing | thực thi front layer → bế tắc → chọn SWAP → cập nhật $\pi$ |
| chỗ cần quyết định | **chọn edge nào để SWAP** - đúng một action |
| $H_{\text{basic}}$ | $\sum_{g \in F} D[\pi(g_1)][\pi(g_2)]$ |
| `basic` / `lookahead` / `decay` | front layer / + extended set / + phạt qubit vừa dùng |
| baseline phải dùng | `SabreSwap(heuristic="decay")`, **nhiều seed** |
| mini_router vs SabreSwap | 10 vs 8 SWAP - cài đúng công thức vẫn thua implementation thật |
| hợp lệ | `CheckMap` → `is_swap_mapped` |
| đúng | bù hoán vị bằng **nghịch đảo** của $\pi$, hoặc chèn SWAP bù vào cuối |
| dao động do seed | ~7% trên mạch 9 qubit - cải thiện nhỏ hơn thế là nhiễu |
| layout vs routing | `SabreLayout` hơn `SabreSwap`+trivial ~16%; phải so cùng layout |
| topology | line 75 → ring 42 → grid 24 → full 0 SWAP |
| swap ≠ depth | hai thước đo không cùng chiều - ảnh hưởng tới thiết kế reward |

### Thử tự làm

1. Sửa `mini_router` để dùng heuristic **lookahead**: cộng thêm
   $W \sum_{g \in E} D[\cdot]$ với $E$ là các cổng ở lớp kế tiếp (gợi ý: sau khi xoá front layer
   khỏi một bản copy của DAG, gọi `front_layer()` lần nữa). Thử $W = 0.5$. Có thắng được
   `SabreSwap basic` không?
2. Thêm hệ số **decay** vào `mini_router`: giữ `decay[q] = 1 + 0.001 * (số lần q bị SWAP)` rồi nhân
   vào $H$. Số SWAP đổi thế nào, depth đổi thế nào?
3. Trong `mini_router`, thay việc chọn SWAP tốt nhất bằng **chọn ngẫu nhiên** trong các ứng viên.
   Chạy 50 lần, vẽ histogram số SWAP. Đây chính là **random policy**, baseline dưới cùng cho RL ở
   notebook `05`.
4. `mini_router` hiện bắt đầu từ layout tầm thường. Cho nó nhận `pi` ban đầu từ `VF2Layout` và đo
   lại. Bao nhiêu phần trăm cải thiện đến từ layout thay vì từ routing?